In [1]:
import sys
import pathlib

# Locate the repo root
_search = pathlib.Path.cwd()
for _ in range(8):
    if (_search / "resourceEstimationPipeline").is_dir():
        REPO_ROOT = str(_search)
        break
    _search = _search.parent
else:
    REPO_ROOT = str(pathlib.Path.cwd())

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4g}".format)

import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ── Pipeline core modules ─────────────────────────────────────────────────────
from resourceEstimationPipeline.config import (
    PipelineConfig,
    TranspileConfig,
    AzureConfig,
    QualtranConfig,
)
from resourceEstimationPipeline.circuit.transpile import (
    transpile_to_clifford_t,
    circuit_stats,
    circuit_to_qasm,
)
from resourceEstimationPipeline.compare.metrics import compare, enrich_from_circuit

# ── Bridge helper (inject Azure params -> Qualtran config) ─────────────────────
from resourceEstimationPipeline.estimators.azure import apply_azure_to_qualtran

# ── Estimators (imported lazily inside run_estimation per circuit) ─────────────

print(f"REPO_ROOT: {REPO_ROOT}")
print("Imports OK")

REPO_ROOT: /Users/kevinli/Documents/resourceEstimation
Imports OK


In [2]:
cfg = PipelineConfig(
    transpile=TranspileConfig(
        optimization_level=1,
        seed_transpiler=42,
        rotation_synthesis_enabled=False,
        rotation_synthesis_epsilon=1e-4,
        synthesis_strategy="qiskit_synth",
        synthesis_method='pygridsynth',
    ),
    azure=AzureConfig(
        error_budget=0.01,
        error_rate=1e-3,
        gate_time_ns=50.0,
        measurement_time_ns=100.0,
        factory_type="RoundBased",
        slow_down_factors=[1.0, 1.5, 2.0],
        optimization_level=1,
        use_graph=False,
        minimize="qubit_hours",
        pareto_index=0,
    ),
    qualtran=QualtranConfig(
        data_d=23,
        phys_err=1e-3,
        error_budget=0.01,
        data_block="fast",
        factory_type="15to1",
        n_factories=6,
        use_gidney_fowler=False,
        use_beverland=True,
        use_azure_parameters=True,
        pareto_index=0,
    ),
)

print(cfg)

PipelineConfig(hamlib=HamlibConfig(hdf5_path='./../hamlib/condensedmatter/heisenberg/heis.hdf5', key=None, key_index=313), evolution=EvolutionConfig(evolution_time=1.0, synthesis_order=2, synthesis_reps=10), transpile=TranspileConfig(basis_gates=['cx', 'rz', 'h', 's', 'sdg', 'x', 'y', 'z', 't', 'tdg'], optimization_level=1, seed_transpiler=42, rotation_synthesis_enabled=False, rotation_synthesis_epsilon=0.0001, synthesis_strategy='qiskit_synth', synthesis_method='pygridsynth', pygridsynth_precision=None), azure=AzureConfig(error_budget=0.01, error_rate=0.001, gate_time_ns=50.0, measurement_time_ns=100.0, two_qubit_gate_time_ns=None, code_distance=None, factory_type='RoundBased', slow_down_factors=[1.0, 1.5, 2.0], optimization_level=1, use_graph=False, minimize='qubit_hours', pareto_index=0), qualtran=QualtranConfig(data_d=23, data_d_sweep=None, phys_err=0.001, t_gate_ns=50.0, t_meas_ns=100.0, cycle_time_us=1.0, error_budget=0.01, data_block='fast', factory_type='15to1', qec_scheme='bev

In [3]:
# ---------------------------------------------------------------------------
# Helper: load a circuit from a .qasm file (optional future extensibility)
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Helper: load a circuit from a .qasm file
# ---------------------------------------------------------------------------

def load_circuit_from_file(path: str):
    """Load a quantum circuit from an OpenQASM file.

    Supports OpenQASM 2.0 and 3.0 files.

    Parameters
    ----------
    path : str
        Path to a .qasm or .qasm3 file

    Returns
    -------
    QuantumCircuit
    """
    from qiskit import QuantumCircuit, qasm3

    if path.endswith(".qasm3"):
        return qasm3.load(path)
    else:
        return QuantumCircuit.from_qasm_file(path)

from pathlib import Path

qasm_dir = Path("qasm3_circuits/225_1000_5000")

c_count, q_count, t_count = map(
    int,
    qasm_dir.name.split("_")
)

print(c_count)  # 225
print(q_count)    # 1000
print(t_count)   # 5000

circuit_list = sorted(
    str(path)
    for path in qasm_dir.glob("*.qasm3")
)

print(f"Found {len(circuit_list)} QASM circuits")
print(circuit_list[:5])

225
1000
5000
Found 225 QASM circuits
['qasm3_circuits/225_1000_5000/1000_1.qasm3', 'qasm3_circuits/225_1000_5000/1000_1072.qasm3', 'qasm3_circuits/225_1000_5000/1000_1429.qasm3', 'qasm3_circuits/225_1000_5000/1000_1786.qasm3', 'qasm3_circuits/225_1000_5000/1000_2143.qasm3']


In [4]:
import re

def _safe_get(result, attr, default=None):
    """Safely retrieve an attribute from a possibly-None result."""
    if result is None:
        return default
    return getattr(result, attr, default)


def transpile_circuit(qc, cfg):
    """Transpile a single circuit to Clifford+T.

    Returns (clifford_t_circuit, stats_dict) or (None, {}) on failure.
    """
    try:
        ct = transpile_to_clifford_t(qc, cfg.transpile)
        stats = circuit_stats(ct)
        return ct, stats
    except Exception as exc:
        print(f"  Warning: Transpilation failed: {exc}")
        return None, {}


def run_estimation(ct_circuit, config):
    """Run one Azure and one Qualtran estimate on a single transpiled circuit.

    Returns (azure_result, qualtran_result) - each may be None on failure.
    Both results are enriched with circuit-derived metrics via enrich_from_circuit().
    """
    azure_result = None
    qualtran_result = None

    # -- Azure QDK ---------------------------------------------------------------
    try:
        from resourceEstimationPipeline.estimators.azure import estimate as azure_estimate
        azure_result = azure_estimate(ct_circuit, config)
        azure_result = enrich_from_circuit(azure_result, ct_circuit)
    except ImportError as e:
        print(f"  Warning: Azure unavailable: {e}")
    except Exception as e:
        print(f"  Warning: Azure estimation failed: {e}")

    # -- Bridge: inject Azure params -> Qualtran config --------------------------
    if azure_result is not None and config.qualtran.use_azure_parameters:
        az_d = _safe_get(azure_result, 'code_distance')
        az_n = _safe_get(azure_result, 'num_factories')
        if (az_d is not None or az_n is not None):
            try:
                config = apply_azure_to_qualtran(azure_result, config)
            except Exception as e:
                print(f"  Warning: Azure->Qualtran bridge failed: {e}")

    # -- Qualtran ----------------------------------------------------------------
    try:
        from resourceEstimationPipeline.estimators.qualtran import estimate as qt_estimate
        qualtran_result = qt_estimate(ct_circuit, config)
        qualtran_result = enrich_from_circuit(qualtran_result, ct_circuit)
    except ImportError as e:
        print(f"  Warning: Qualtran unavailable: {e}")
    except Exception as e:
        print(f"  Warning: Qualtran estimation failed: {e}")

    return azure_result, qualtran_result


# ---------------------------------------------------------------------------
# Simplify estimator names for plotting.
# The full name (with config params) is preserved in the DataFrame; this helper
# only extracts the family so that all points from the same estimator produce
# a single continuous line on each plot.
# ---------------------------------------------------------------------------

_FAMILY_RE = re.compile(r"^(Azure|Qualtran)")


def _family(name: str) -> str:
    """Return 'Azure' or 'Qualtran', falling back to the full name."""
    m = _FAMILY_RE.search(name)
    return m.group(1) if m else name


# ---------------------------------------------------------------------------
# collect_metrics: one row per (circuit, estimator) — same format as
# comparison_simple.ipynb but extended with qubit breakdown columns.
# ---------------------------------------------------------------------------

def collect_metrics(circuit_name, azure_result, qualtran_result, ct_stats):
    """Collect metrics into DataFrame rows.

    Keeps the side-by-side format from comparison_simple.ipynb:
      Metric | Azure (d=...,...) | Qualtran (d=...,...) | Ratio
    for every circuit, plus qubit breakdown columns for plotting.

    Missing fields become None -> pandas renders them blank.
    """
    for result in [azure_result, qualtran_result]:
        if result is None:
            continue
        t_count = _safe_get(result, 't_count')
        runtime = _safe_get(result, 'runtime_seconds')
        total_q = _safe_get(result, 'physical_qubits')
        compute_q = _safe_get(result, 'physical_compute_qubits')
        factory_q = _safe_get(result, 'physical_factory_qubits')

        # Space-time volume (qubit-seconds) using the same units each estimator reports.
        if total_q is not None and runtime is not None:
            space_time = float(total_q) * runtime
        else:
            space_time = None

        yield {
            "circuit_name":       circuit_name,
            "estimator_family":   _family(result.estimator_name),
            "t_count":            t_count,
            "clifford_count":     _safe_get(result, 'clifford_count'),
            "rotation_count":     _safe_get(result, 'rotation_count'),
            "toffoli_count":      _safe_get(result, 'toffoli_count'),
            "measurement_count":  _safe_get(result, 'measurement_count'),
            "runtime_seconds":    runtime,
            "total_qubits":       total_q,
            "compute_qubits":     compute_q,
            "factory_qubits":     factory_q,
            "space_time_volume":  space_time,
            "code_distance":      _safe_get(result, 'code_distance'),
            "logical_error_rate": _safe_get(result, 'logical_error_rate'),
            "error_budget":       _safe_get(result, 'error_budget'),
            "physical_error_rate":_safe_get(result, 'physical_error_rate'),
            "logical_qubits":     _safe_get(result, 'logical_qubits'),
            "logical_cycles":     _safe_get(result, 'logical_cycles'),
            "factory_count":      _safe_get(result, 'factory_count'),
            "num_factories":      _safe_get(result, 'num_factories'),
        }


def run_benchmark(circuit_list, config):
    """Run the full multi-circuit benchmarking pipeline.

    Parameters
    ----------
    circuit_list : list[QuantumCircuit] or list[str]  circuits or file paths
    config       : PipelineConfig

    Returns
    -------
    (pandas.DataFrame, dict[str, dict[str, EstimationResult]])
        DataFrame: one row per (circuit, estimator) — same format as
        comparison_simple.ipynb extended with qubit breakdown columns.
        circuit_results: nested dict {circuit_name: {'Azure': result, 'Qualtran': result}}
            so the side-by-side section can use the real EstimationResult objects.

        Columns for plotting:
            circuit_name, estimator_family, t_count, total_qubits,
            compute_qubits, factory_qubits, space_time_volume, runtime_seconds
        Columns from the comparison layer:
            clifford_count, rotation_count, logical_error_rate, code_distance, etc.
    """
    all_rows = []
    circuit_results = {}  # {circuit_name: {'Azure': result, 'Qualtran': result}}

    for idx, item in enumerate(circuit_list):
        # Derive circuit name
        if isinstance(item, str):
            circuit_name = pathlib.Path(item).stem
        elif hasattr(item, 'name') and getattr(item, 'name', None):
            circuit_name = item.name
        else:
            circuit_name = f"circuit_{idx}"

        # Step A: load (if path string) -> transpile
        qc = load_circuit_from_file(item) if isinstance(item, str) else item
        ct_circuit, ct_stats = transpile_circuit(qc, config)
        if ct_circuit is None:
            print(f"  Skipping circuit '{circuit_name}' (transpilation failed).")
            continue

        print(f"[{idx+1}/{len(circuit_list)}] Circuit '{circuit_name}': "
              f"T={ct_stats.get('t_count', '?')}, qubits={ct_stats.get('num_qubits', '?')}")

        # Step B: estimate (one Azure + one Qualtran)
        azure_res, qualtran_res = run_estimation(ct_circuit, config)

        # Save real EstimationResult objects for side-by-side comparison.
        circuit_results[circuit_name] = {}
        if azure_res is not None:
            circuit_results[circuit_name]['Azure'] = azure_res
        if qualtran_res is not None:
            circuit_results[circuit_name]['Qualtran'] = qualtran_res

        # Step C: collect metrics into DataFrame rows
        rows = list(collect_metrics(circuit_name, azure_res, qualtran_res, ct_stats))
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)
    return df, circuit_results

In [5]:
import time

start_time = time.time()
benchmark_df, circuit_results = run_benchmark(circuit_list, cfg)
end_time = time.time()
exec_time = end_time - start_time
print(f"\nBenchmark complete ({exec_time:.4f} seconds). {len(benchmark_df)} rows collected.")

# -- Display side-by-side comparison for the first circuit -------------------
# Uses the real EstimationResult objects saved during benchmarking — the same
# approach as comparison_simple.ipynb (cells with report, comparison_dataframe, etc.).
from resourceEstimationPipeline.compare.metrics import compare
from resourceEstimationPipeline.compare.tables import (
    comparison_dataframe, differences_dataframe, missing_dataframe, explain_differences,
)

estimator_names_in_df = benchmark_df['estimator_family'].unique()
circuits_list = benchmark_df['circuit_name'].unique()

if len(circuits_list) > 0 and len(estimator_names_in_df) >= 2:
    first_circuit = circuits_list[0]

    # Pull the real EstimationResult objects (not fake wrappers).
    results_map = circuit_results.get(first_circuit, {})
    azure_r = results_map.get('Azure')
    qt_r = results_map.get('Qualtran')

    available = [r for r in [azure_r, qt_r] if r is not None]

    if len(available) >= 2:
        # Build comparison report using the same helper as comparison_simple.ipynb.
        report = compare(available)
        print(f"\nCircuit : {first_circuit}")
        print(f"Estimators compared   : {report.estimator_names}")
        print(f"Shared metrics        : {len(report.shared_metrics)}")
        print(f"N/A in ≥1 estimator   : {len(report.missing_metrics)}")
        print(f"Numeric differences   : {len(report.differences)}")

        # Full comparison table — identical format to comparison_simple.ipynb.
        n_total = len(report.metric_rows)
        display(Markdown(
            f"*{n_total} metrics total — "
            f"**{len(report.shared_metrics)} shared** | "
            f"**{len(report.missing_metrics)} framework-specific*.*"
        ))
        df_comp = comparison_dataframe(report)
        display(df_comp)

        # Metrics that differ between estimators.
        diff_df = differences_dataframe(report)
        if not diff_df.empty:
            print("\nMetrics that differ:")
            display(diff_df)

        # Explanation of differences (same text as comparison_simple.ipynb).
        print(explain_differences(report))
    else:
        print("Need both Azure and Qualtran results for comparison.")
else:
    print("Not enough data for a side-by-side table.")

[1/225] Circuit '1000_1': T=1, qubits=1000
[2/225] Circuit '1000_1072': T=1072, qubits=1000
[3/225] Circuit '1000_1429': T=1429, qubits=1000
[4/225] Circuit '1000_1786': T=1786, qubits=1000
[5/225] Circuit '1000_2143': T=2143, qubits=1000
[6/225] Circuit '1000_2500': T=2500, qubits=1000
[7/225] Circuit '1000_2858': T=2858, qubits=1000
[8/225] Circuit '1000_3215': T=3215, qubits=1000
[9/225] Circuit '1000_3572': T=3572, qubits=1000
[10/225] Circuit '1000_358': T=358, qubits=1000
[11/225] Circuit '1000_3929': T=3929, qubits=1000
[12/225] Circuit '1000_4286': T=4286, qubits=1000
[13/225] Circuit '1000_4643': T=4643, qubits=1000
[14/225] Circuit '1000_5000': T=5000, qubits=1000
[15/225] Circuit '1000_715': T=715, qubits=1000
[16/225] Circuit '145_1': T=1, qubits=145
[17/225] Circuit '145_1072': T=1072, qubits=145
[18/225] Circuit '145_1429': T=1429, qubits=145
[19/225] Circuit '145_1786': T=1786, qubits=145
[20/225] Circuit '145_2143': T=2143, qubits=145
[21/225] Circuit '145_2500': T=2500

*39 metrics total — **35 shared** | **4 framework-specific*.*

,Metric,"Azure QDK (err_rate=1e-03, budget=0.01)","Qualtran (d=9, p=1e-03)",Ratio (B/A)
0,Logical qubits,"1,000","1,000",1.000×
1,Logical depth,3,3,1.000×
2,Logical cycles,1,6.111,6.111×
3,T count,1,1,1.000×
4,T depth,1,1,1.000×
5,T count (from circuit),1,1,1.000×
6,Clifford count,2,2,1.000×
7,Rotation count,0,0,—
8,Toffoli count,0,0,—
9,Measurement count,0,0,—



Metrics that differ:


,Metric,"Azure QDK (err_rate=1e-03, budget=0.01)","Qualtran (d=9, p=1e-03)",Ratio (B/A)
0,Logical cycles,1,6.111,6.111×
1,Physical qubits (total),"336,711","341,208",1.013×
2,Physical compute qubits,"336,651","338,742",1.006×
3,Physical factory qubits,60,"2,466",41.100×
4,Runtime (s),3.6e-06,2.2e-05,6.111×
5,Space-Time (qubit s),1.212,7.507,6.193×
6,Logical error rate,0.007727,0.03455,4.472×
7,Physical qubits per logical qubit,336.7,341.2,1.013×
8,Factory qubit fraction,0.0001782,0.007227,40.558×
9,Runtime per T gate (s),3.6e-06,2.2e-05,6.111×


WHY DO THE ESTIMATORS PRODUCE DIFFERENT RESULTS?

Both estimators receive the SAME canonical Clifford+T circuit, so any differences arise purely from the resource estimation models, not the input circuit.

ROTATION CONTEXT:
  • Azure QDK (err_rate=1e-03, budget=0.01): genuinely no Rz rotations — the input circuit has zero arbitrary-angle Rz gates. t_per_rotation is meaningless (reported as N/A).
  • Qualtran (d=9, p=1e-03): genuinely no Rz rotations — the input circuit has zero arbitrary-angle Rz gates. t_per_rotation is meaningless (reported as N/A).

Key model differences:

1. T-gate synthesis for arbitrary Rz rotations
   • Azure QDK: reports NUM_TS_PER_ROTATION (the actual synthesis count
     it uses internally for the given error budget).
   • Qualtran: counts raw Rz gates from the bloq graph; T synthesis cost
     is estimated post-hoc via the Solovay-Kitaev formula ~3·log₂(1/ε).
   → Expect T-count differences when rotation_count > 0. When rotation_count == 0
     both estimato

### Results table

Each row is one (circuit, estimator) pair.  Empty cells indicate unavailable metrics.


In [6]:
benchmark_df

,circuit_name,estimator_family,t_count,clifford_count,rotation_count,toffoli_count,measurement_count,runtime_seconds,total_qubits,compute_qubits,factory_qubits,space_time_volume,code_distance,logical_error_rate,error_budget,physical_error_rate,logical_qubits,logical_cycles,factory_count,num_factories
0,1000_1,Azure,1,2,0,0,0,3.6e-06,336711,336651,60,1.212,9,0.007727,0.01,0.001,1000,1,1×T,1
1,1000_1,Qualtran,1,2,0,0,0,2.2e-05,341208,338742,2466,7.507,9,0.03455,0.01,0.001,1000,6.111,"15to1×1 (2,466 qubits)",1
2,1000_1072,Azure,1072,2144,0,0,0,0.005574,756187,704667,51520,4215,13,0.009046,0.01,0.001,1000,1072,8×T,8
3,1000_1072,Qualtran,1072,2144,0,0,0,0.005574,747734,706758,40976,4168,13,0.08824,0.01,0.001,1000,1072,"15to1×8 (5,122 each)",8
4,1000_1429,Azure,1429,2858,0,0,0,0.007431,801067,704667,96400,5953,13,0.009319,0.01,0.001,1000,1429,10×T,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445,929_4643,Qualtran,4643,9286,0,0,0,0.02786,936990,875700,61290,2.61e+04,15,0.04111,0.01,0.001,929,4643,"15to1×9 (6,810 each)",9
446,929_5000,Azure,5000,10000,0,0,0,0.03,960514,873754,86760,2.882e+04,15,0.004159,0.01,0.001,929,5000,9×T,9
447,929_5000,Qualtran,5000,10000,0,0,0,0.03,936990,875700,61290,2.811e+04,15,0.04427,0.01,0.001,929,5000,"15to1×9 (6,810 each)",9
448,929_715,Azure,715,1430,0,0,0,0.003718,707322,655802,51520,2630,13,0.005722,0.01,0.001,929,715,8×T,8


In [7]:
from pathlib import Path

results_dir = (
    Path("results")
    / f"{c_count}_{q_count}_{t_count}"
)

results_dir.mkdir(parents=True, exist_ok=True)

benchmark_df.to_csv(
    results_dir / "benchmark_results.csv",
    index=False,
)

In [8]:
azure = benchmark_df[
    benchmark_df["estimator_family"]=="Azure"
].copy()

qualtran = benchmark_df[
    benchmark_df["estimator_family"]=="Qualtran"
].copy()


ratio_df = pd.merge(
    azure,
    qualtran,
    on=[
        "circuit_name",
        "t_count",
        "logical_qubits",
    ],
    suffixes=("_azure", "_qualtran")
)

print(ratio_df.shape)

(225, 37)


In [9]:
ratio_df["total_ratio"] = (
    ratio_df["total_qubits_qualtran"]
    /
    ratio_df["total_qubits_azure"]
)


ratio_df["compute_ratio"] = (
    ratio_df["compute_qubits_qualtran"]
    /
    ratio_df["compute_qubits_azure"]
)


ratio_df["factory_ratio"] = (
    ratio_df["factory_qubits_qualtran"]
    /
    ratio_df["factory_qubits_azure"]
)

In [10]:
ratio_df.to_csv(
    results_dir / "ratio_results.csv",
    index=False,
)

# Heat Map Actual Values

In [81]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

def plot_ratio_heatmap(
    df,
    ratio_column,
    title,
    filename=None,
):

    # Create heatmap directly from actual values
    heatmap_data = (
        df.groupby(
            ["logical_qubits", "t_count"]
        )[ratio_column]
        .mean()
        .unstack()
    )


    fig, ax = plt.subplots(
        figsize=(20,11),
        dpi=150
    )

    # Ensure low qubit counts are at the bottom
    heatmap_data = heatmap_data.sort_index(ascending=True)

    # Ensure low T counts are on the left
    heatmap_data = heatmap_data.sort_index(axis=1, ascending=True)

    # Mask extreme values
    threshold = 3.0

    masked_data = heatmap_data.mask(
        heatmap_data > threshold
    )

    # Compute color limits only from valid values
    vmin = masked_data.min().min()
    vmax = masked_data.max().max()

    sns.heatmap(
        heatmap_data,
        cmap="coolwarm",
        center=1,
        vmin=vmin,
        vmax=vmax,
        annot=True,
        fmt=".3f",
        annot_kws={"fontsize": 10},
        linewidths=0,          # remove gaps
        linecolor=None,
        square=True,           # make each cell square
        cbar_kws={
            "label": "Qualtran / Azure"
        },
        ax=ax
    )

    # Overlay masked cells in black
    mask = heatmap_data > threshold

    for y, row in enumerate(mask.values):
        for x, value in enumerate(row):
            if value:
                ax.add_patch(
                    plt.Rectangle(
                        (x, y),
                        1,
                        1,
                        fill=True,
                        color="black"
                    )
                )

    ax.invert_yaxis()

    ax.set_xlabel(
        "T Count"
    )

    ax.set_ylabel(
        "Logical Qubits"
    )

    ax.set_title(
        title
    )

    ax.set_xticklabels(
        heatmap_data.columns,
        rotation=45,
        ha="right"
    )

    ax.set_yticklabels(
        heatmap_data.index,
        rotation=0
    )


    plt.tight_layout()


    if filename:
        plt.savefig(
            filename,
            bbox_inches="tight",
            dpi=150
        )
        plt.close()
    else:
        plt.show()

    return fig

In [82]:
from pathlib import Path

# Create directory
output_dir = Path("heat_map") / f"{c_count}_{q_count}_{t_count}"
output_dir.mkdir(parents=True, exist_ok=True)

# File paths
total_path = output_dir / "total_qubit_ratio_heatmap.png"
compute_path = output_dir / "compute_qubit_ratio_heatmap.png"
factory_path = output_dir / "factory_qubit_ratio_heatmap.png"

fig1 = plot_ratio_heatmap(
    ratio_df,
    "total_ratio",
    "Total Physical Qubit Ratio\nQualtran / Azure",
    total_path,
    # x_bins=int(math.sqrt(c_count)),
    # y_bins=int(math.sqrt(c_count))
)

fig2 = plot_ratio_heatmap(
    ratio_df,
    "compute_ratio",
    "Compute Qubit Ratio\nQualtran / Azure",
    compute_path,
    # x_bins=int(math.sqrt(c_count)),
    # y_bins=int(math.sqrt(c_count))
)

fig3 = plot_ratio_heatmap(
    ratio_df,
    "factory_ratio",
    "Factory Qubit Ratio\nQualtran / Azure",
    factory_path,
    # x_bins=int(math.sqrt(c_count)),
    # y_bins=int(math.sqrt(c_count))
)

In [13]:
import json
from dataclasses import asdict

with open(results_dir / "config.json", "w") as f:
    json.dump(asdict(cfg), f, indent=4)